In [1]:
import pandas as pd
import requests
import numpy as np
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
#url de la API
api_url='https://api.yelp.com/v3/businesses/search'

#estos datos corresponden a una cuenta de usuario creada previamente
clientid='GWOCZh9-BmZxtdsAjr7Gug'
apikey='FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx'
headers={'Authorization':'Bearer %s'%apikey}

In [4]:
ciudad = "Miami" # Se puede elegir cualquier ciudad del dataset entregado

In [5]:
# De esta manera se consulta el API de Yelp para obtener información sobre restaurantes en la ciudad especificada
params={'term':'restaurants','location': ciudad,'limit':50}
response=requests.get(api_url,params=params,headers=headers)

data=response.json()
data

{'businesses': [{'id': 'UXHxLN3DcDGI57uDIfCuJA',
   'alias': 'olds-havana-cuban-bar-and-cocina-miami',
   'name': "Old's Havana Cuban Bar & Cocina",
   'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/OyMD-xvBjobfDmDdBs2Jfw/o.jpg',
   'is_closed': False,
   'url': 'https://www.yelp.com/biz/olds-havana-cuban-bar-and-cocina-miami?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug',
   'review_count': 3189,
   'categories': [{'alias': 'cuban', 'title': 'Cuban'},
    {'alias': 'bars', 'title': 'Bars'},
    {'alias': 'venues', 'title': 'Venues & Event Spaces'}],
   'rating': 4.4,
   'coordinates': {'latitude': 25.765607074496565,
    'longitude': -80.21921771559332},
   'transactions': ['restaurant_reservation', 'pickup'],
   'price': '$$',
   'location': {'address1': '1442 SW 8th St',
    'address2': '',
    'address3': None,
    'city': 'Miami',
    'zip_code': '33135',
    'country': 'US',
    'state': 

In [6]:
#Veamos los keys del diccionario recibido
data.keys()

dict_keys(['businesses', 'total', 'region'])

In [7]:
#El primer elemento del diccionario indica el total de restaurants existentes en la API
print('En total la base de datos registra %d restaurants'%data['total'])

En total la base de datos registra 5700 restaurants


In [8]:
data['businesses'] # Hasta acá se le entregaría a los estudiantes

[{'id': 'UXHxLN3DcDGI57uDIfCuJA',
  'alias': 'olds-havana-cuban-bar-and-cocina-miami',
  'name': "Old's Havana Cuban Bar & Cocina",
  'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/OyMD-xvBjobfDmDdBs2Jfw/o.jpg',
  'is_closed': False,
  'url': 'https://www.yelp.com/biz/olds-havana-cuban-bar-and-cocina-miami?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug',
  'review_count': 3189,
  'categories': [{'alias': 'cuban', 'title': 'Cuban'},
   {'alias': 'bars', 'title': 'Bars'},
   {'alias': 'venues', 'title': 'Venues & Event Spaces'}],
  'rating': 4.4,
  'coordinates': {'latitude': 25.765607074496565,
   'longitude': -80.21921771559332},
  'transactions': ['restaurant_reservation', 'pickup'],
  'price': '$$',
  'location': {'address1': '1442 SW 8th St',
   'address2': '',
   'address3': None,
   'city': 'Miami',
   'zip_code': '33135',
   'country': 'US',
   'state': 'FL',
   'display_address': ['1442 S

# Transformacion

In [9]:
df_yelp_v1 = pd.DataFrame(data['businesses'])
# numero de elementos
print('El número de restaurantes obtenidos es: %d'%len(df_yelp_v1))

El número de restaurantes obtenidos es: 50


In [10]:
contador = 0
lista_consolidada = []

while contador < 200:
    params={'term':'restaurants','location': ciudad,'limit':50,'offset':contador}
    response=requests.get(api_url,params=params,headers=headers)
    data=response.json()

    df_yelp = pd.json_normalize(
        data['businesses'],
        record_path='categories',
        meta=[
            'id','name','review_count','rating','price',
            ['coordinates','latitude'],
            ['coordinates','longitude'],
            ['location','address1']
        ],
        sep='_',
        errors='ignore'
    )

    lista_consolidada.append(df_yelp)
    contador += 50
    


In [11]:
df_api_yelp_v2 = pd.concat(lista_consolidada, ignore_index=True)

In [17]:
display(df_api_yelp_v2.head(10))

,alias,title,id,name,review_count,rating,price,coordinates_latitude,coordinates_longitude,location_address1
0,cuban,Cuban,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3189,4.4,$$,25.765607,-80.219218,1442 SW 8th St
1,bars,Bars,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3189,4.4,$$,25.765607,-80.219218,1442 SW 8th St
2,venues,Venues & Event Spaces,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3189,4.4,$$,25.765607,-80.219218,1442 SW 8th St
3,asianfusion,Asian Fusion,K3ukx2e11xTRtYBU01dmrA,Salty Flame,259,4.3,NaN,25.76022,-80.19267,1414 Brickell Ave
4,steak,Steakhouses,K3ukx2e11xTRtYBU01dmrA,Salty Flame,259,4.3,NaN,25.76022,-80.19267,1414 Brickell Ave
5,cocktailbars,Cocktail Bars,K3ukx2e11xTRtYBU01dmrA,Salty Flame,259,4.3,NaN,25.76022,-80.19267,1414 Brickell Ave
6,mediterranean,Mediterranean,ix8ifP1jQM9ektdVAs19sQ,Crazy About You,2881,4.2,$$$,25.762015,-80.188986,1155 Brickell Bay Dr
7,italian,Italian,ix8ifP1jQM9ektdVAs19sQ,Crazy About You,2881,4.2,$$$,25.762015,-80.188986,1155 Brickell Bay Dr
8,spanish,Spanish,ix8ifP1jQM9ektdVAs19sQ,Crazy About You,2881,4.2,$$$,25.762015,-80.188986,1155 Brickell Bay Dr
9,italian,Italian,oxtMfBGmVNE18pFVuw7lFg,Fratellino,1893,4.8,$$$,25.74917,-80.26011,264 Miracle Mile


In [13]:
print(f'El número de restaurantes obtenidos es: {len(df_yelp)}')
print(f'El número de restaurantes obtenidos es: {len(df_api_yelp_v2)}')

El número de restaurantes obtenidos es: 107
El número de restaurantes obtenidos es: 438


In [18]:
#exportar csv
df_api_yelp_v2.to_csv('restaurante_api_yelp_miami.csv', index=False)

In [16]:
# valores_unicos = df_api_yelp_v2['alias'].unique()
# print(valores_unicos)